In [1]:
import torch
import torch.nn as nn
from transformers import AutoModelForTokenClassification

In [2]:
class QASpanDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = AutoModelForTokenClassification.from_pretrained('bert-base-uncased', num_labels=2)

    def forward(self, data):
        outputs = self.model(input_ids=data['input_ids'],
                             attention_mask=data['attention_mask'])
        return outputs

In [3]:
model = QASpanDetector()

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForTokenClassification: ['cls.predictions.decoder.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.weight', 'cls.predictions.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-u

In [4]:
import pickle

with open('data/debug_preprocessed.pkl', 'rb') as f:
    train_encodings = pickle.load(f)

In [5]:
len(train_encodings['input_ids'])

753

In [6]:
len(train_encodings['input_ids'][0])

451

In [7]:
from preprocess import SquadDataset

train_dataset = SquadDataset(train_encodings)

In [8]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=False)

In [9]:
import torch.nn.functional as F

def get_loss(reg_pred, obj_pred, reg_target, obj_target, answer_start):
    # Get answer length by get the the value at the answer start index of predict and target vector
    reg_pred = torch.gather(reg_pred, 1, answer_start).exp()
    reg_target = torch.gather(reg_target, 1, answer_start)

    reg_loss = F.mse_loss(reg_pred, reg_target)
    obj_loss = F.binary_cross_entropy_with_logits(obj_pred, obj_target)

    return reg_loss + obj_loss
    

In [10]:
MODEL_MAX_LENGTH = 512

In [16]:
from torch.optim import AdamW

torch.manual_seed(0)
epochs = 3
print_every = 20
optim = AdamW(model.parameters(), lr=5e-5)
for epoch in range(epochs):
    # Set model in train mode
    model.train()
    loss_of_epoch = 0

    print("############Train############")
    for batch_idx, batch in enumerate(train_loader):
        sentence_length = batch['input_ids'].size(1)

        answer_start = batch['start_positions'].reshape(train_loader.batch_size, 1)        
        attention_mask = batch['attention_mask']
        attention_mask = F.pad(attention_mask, (0, MODEL_MAX_LENGTH - sentence_length), 'constant', 0)

        reg_target = batch['labels'][:, :, 1]
        obj_target = batch['labels'][:, :, 0]
        obj_target = obj_target.type(torch.FloatTensor)
        reg_target = reg_target.type(torch.FloatTensor)

        outputs = model(batch)

        reg_predict = outputs['logits'][:, :, 1]
        obj_predict = outputs['logits'][:, :, 0]
        reg_predict = F.pad(reg_predict, (0, MODEL_MAX_LENGTH - sentence_length), 'constant', 0)
        obj_predict = F.pad(obj_predict, (0, MODEL_MAX_LENGTH - sentence_length), 'constant', 0)
        obj_predict = obj_predict * attention_mask
        
        loss = get_loss(reg_predict, obj_predict, reg_target, obj_target, answer_start)
        # loss.backward()
        # optim.step()
        loss_of_epoch += loss.item()
        if (batch_idx + 1) % print_every == 0:
            print("Batch {:} / {:}".format(batch_idx + 1, len(train_loader)))
            print("Loss:", round(loss.item(), 1), "\n")
        break
    break

############Train############
Batch 0 / 95
Loss: 7.0 

